# 1. Import library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE

import warnings
warnings.filterwarnings('ignore')

print("Cell 1: Semua Library Berhasil Dimuat!")


# 2. Load dataset

In [ ]:
file_path = '../data/WA_Fn-UseC_-Telco-Customer-Churn.csv' 
train_df = pd.read_csv(file_path)

print("Dataset Telco Churn Berhasil Dimuat!")
print(f"Dimensi Data: {train_df.shape[0]} Baris, {train_df.shape[1]} Kolom")
print(f"\nDistribusi Target:\n{train_df['Churn'].value_counts()}")
print(f"\nPersentase Churn:\n{train_df['Churn'].value_counts(normalize=True)*100}")

# 3. Ambil feature & label

In [ ]:
# Drop ID karena tidak relevan untuk ML
train_df = train_df.drop(columns=['customerID'])

# Memisahkan X dan y
X = train_df.drop(columns=['Churn'])
y = train_df['Churn'].map({'Yes': 1, 'No': 0}) # Ubah ke angka untuk evaluasi

print("Feature (X) dan Label (y) Berhasil Dipisahkan!")
print(f"Dimensi Feature (X): {X.shape[0]} Baris, {X.shape[1]} Kolom")
print(f"Dimensi Label (y): {y.shape[0]} Baris")

# 4. Split dataset

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Dataset Berhasil Di-split (80% Train, 20% Test)!")
print(f"Dimensi X_train: {X_train.shape[0]} Baris")
print(f"Dimensi X_test: {X_test.shape[0]} Baris")

# 5. Preprocessing

In [ ]:
# 1. Handling Missing Values di TotalCharges (Ubah spasi jadi NaN, lalu isi median)
X_train['TotalCharges'] = pd.to_numeric(X_train['TotalCharges'], errors='coerce')
X_test['TotalCharges'] = pd.to_numeric(X_test['TotalCharges'], errors='coerce')

median_val = X_train['TotalCharges'].median()
X_train['TotalCharges'] = X_train['TotalCharges'].fillna(median_val)
X_test['TotalCharges'] = X_test['TotalCharges'].fillna(median_val)

# 2. Encoding (One-Hot Encoding untuk data teks)
X_train_encoded = pd.get_dummies(X_train, drop_first=True)
X_test_encoded = pd.get_dummies(X_test, drop_first=True)

# Menyamakan kolom jika ada kategori yang hilang saat di-split
X_train_encoded, X_test_encoded = X_train_encoded.align(X_test_encoded, join='left', axis=1, fill_value=0)

# 3. Scaling (Wajib ada untuk SVM)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_encoded)
X_test_scaled = scaler.transform(X_test_encoded)

print("Preprocessing Selesai!")
print("✓ Handling Missing Values (TotalCharges diisi median)")
print("✓ Encoding Kategorikal (One-Hot Encoding)")
print("✓ Scaling Data (StandardScaler)")
print(f"Dimensi X_train_scaled: {X_train_scaled.shape[0]} Baris, {X_train_scaled.shape[1]} Kolom")

# 6. SMOTE + classifier

In [ ]:
# Menerapkan SMOTE
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

# Melatih Classifier SVM Base (Sebagai perbandingan awal)
svm_base = SVC(random_state=42)
svm_base.fit(X_train_smote, y_train_smote)

print("SMOTE Berhasil Diterapkan!")
print(f"Distribusi Target Sesudah SMOTE:\n{y_train_smote.value_counts()}")
print("\nClassifier SVM (Base Model) Berhasil Dilatih dengan Data SMOTE!")

# 7. Training + GridSearchCV (HPO)

In [ ]:
import time
print("Memulai Hyperparameter Optimization (GridSearchCV)...")
start_time = time.time()

# Kita batasi grid-nya sedikit agar tidak hang (hapus C=10 karena biasanya komputasi C besar sangat lama di SVM)
param_grid = {
    'C': [0.1, 1], 
    'kernel': ['linear', 'rbf']
}

# RAHASIA KECEPATAN: probability=True KITA HAPUS DULU DI SINI
grid_search = GridSearchCV(SVC(random_state=42), param_grid, cv=3, scoring='accuracy', n_jobs=-1)

# Mulai pencarian parameter (Ini mungkin butuh 1-3 menit, ditunggu saja ya)
grid_search.fit(X_train_smote, y_train_smote)

print(f"Training HPO Selesai dalam {(time.time() - start_time)/60:.2f} menit!")
print(f"Parameter SVM Terbaik: {grid_search.best_params_}")

# Latih ulang 1x model final MENGGUNAKAN probability=True untuk kebutuhan UI
print("\nMelatih ulang model final dengan parameter terbaik untuk Web UI...")
best_params = grid_search.best_params_
best_svm = SVC(**best_params, probability=True, random_state=42)
best_svm.fit(X_train_smote, y_train_smote)

print("Model Final (best_svm) Siap Digunakan!")

# 8. Akurasi + threshold

In [ ]:
# Prediksi Data Mentah (Imbalanced) untuk perbandingan dosen
svm_imbalanced = SVC(random_state=42).fit(X_train_scaled, y_train)
y_pred_imb = svm_imbalanced.predict(X_test_scaled)
acc_imb = accuracy_score(y_test, y_pred_imb)

# Prediksi Data Balanced (SMOTE + HPO)
y_pred_final = best_svm.predict(X_test_scaled)
acc_final = accuracy_score(y_test, y_pred_final)

print("=== HASIL AKURASI MODEL ===")
print(f"Akurasi Model Data Mentah (Imbalanced): {acc_imb * 100:.2f}%")
print(f"Akurasi Model Final (SMOTE + HPO): {acc_final * 100:.2f}%")

# 9. Confusion matrix

In [ ]:
cm_final = confusion_matrix(y_test, y_pred_final)

plt.figure(figsize=(6,4))
sns.heatmap(cm_final, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Churn (0)', 'Churn (1)'],
            yticklabels=['No Churn (0)', 'Churn (1)'])
plt.title('Confusion Matrix - Model Final SVM (SMOTE + HPO)')
plt.ylabel('Kondisi Sebenarnya')
plt.xlabel('Tebakan Model')
plt.show()

# 10. Detail perbandingan

In [ ]:
print("=== DETAIL PERBANDINGAN (CLASSIFICATION REPORT) ===")
print("1. HASIL MODEL MENTAH (IMBALANCED)")
print(classification_report(y_test, y_pred_imb))

print("-" * 55)

print("2. HASIL MODEL FINAL (SMOTE + HPO)")
print(classification_report(y_test, y_pred_final))

# 11. Simpan model

In [ ]:
if not os.path.exists('models'):
    os.makedirs('models')

# Simpan Model, Scaler, dan urutan kolom
with open('models/svm_model.pkl', 'wb') as f:
    pickle.dump(best_svm, f)
with open('models/svm_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
with open('models/X_columns.pkl', 'wb') as f:
    pickle.dump(X_train_encoded.columns.tolist(), f)

print("Proses Simpan Berhasil!")
print("✅ svm_model.pkl")
print("✅ svm_scaler.pkl")
print("✅ X_columns.pkl (Siap dicolok ke Backend / UI)")

# 12. Sanity check

In [ ]:
print("Melakukan Sanity Check pada 1 data pelanggan acak...")

# Load kembali file pkl
with open('models/svm_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)
with open('models/svm_scaler.pkl', 'rb') as f:
    loaded_scaler = pickle.load(f)

# Simulasi data yang diinput lewat UI
dummy_data = X_test_encoded.iloc[[0]] 
dummy_scaled = loaded_scaler.transform(dummy_data)

# Prediksi
prediksi = loaded_model.predict(dummy_scaled)
probabilitas = loaded_model.predict_proba(dummy_scaled)[0]

hasil_teks = "CHURN (Bakal Pindah)" if prediksi[0] == 1 else "TIDAK CHURN (Menetap)"

print("\n=== HASIL SANITY CHECK ===")
print(f"Tebakan Model: {hasil_teks}")
print(f"Tingkat Keyakinan: {probabilitas[prediksi[0]] * 100:.2f}%")
print("✅ Sanity Check Sukses! SVM siap dipresentasikan.")